# Pipeline: Embedding → Retrieval → Generation → Evaluation

This notebook implements our RAG pipeline:

| Component | What it does | Our approaches |
|-----------|-------------|----------------|
| **Embedding** | Convert chunks to vectors | A: MiniLM (384d), B: MPNet (768d), C: MPNet + metadata prefix |
| **Retrieval** | Find top 5 relevant chunks | 1: Cosine similarity, 2: Cosine + cross-encoder re-ranking, 3: BM25 keywords |
| **Generation** | Produce answers with LLM | A: Simple prompt (XML), B: Guided prompt, C: Few-shot prompt D: Chain of Thought (CoT) |
| **Evaluation** | Evaluate the responses | Recall@5, MRR, Token F1, ROUGE L, BERTScore


We test all combinations: **3 chunk sets, 3 embeddings, 3 retrieval, 4 prompts**

Each configuration is tested on the benchmark RAG.

In [55]:
!pip install --quiet sentence-transformers transformers accelerate rank-bm25 numpy scikit-learn torch rouge-score


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import json
import os
import time
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

C:\Users\Hermes\anaconda3\envs\cv-cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA GeForce RTX 3070


In [3]:
import os
import glob
import json

print("loading JSONL files...")

jsonl_files = glob.glob("*.jsonl") 

all_chunks = []

for file_path in jsonl_files:
    print(f"load: {file_path}")
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            
            if line.strip(): 
                chunk_data = json.loads(line)
                all_chunks.append(chunk_data)

print(f"total: {len(all_chunks)} chunks")

loading JSONL files...
load: section-aware_chunks.jsonl
load: sliding_fixed_chunks.jsonl
load: sliding_sentence_chunks.jsonl
total: 2994 chunks


In [4]:
CHUNK_FILES = {
    "section_aware": "section-aware_chunks.jsonl",
    "sliding_fixed": "sliding_fixed_chunks.jsonl",
    "sliding_sentence": "sliding_sentence_chunks.jsonl",
}

def load_chunks(filepath):
    """Read a JSONL file and return a list of chunk dicts."""
    chunks = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                chunks.append(json.loads(line))
    return chunks

# load each chunk set
chunk_sets = {}
for name, filepath in CHUNK_FILES.items():
    if os.path.exists(filepath):
        chunk_sets[name] = load_chunks(filepath)
        print(f"Loaded {name}: {len(chunk_sets[name])} chunks")
    else:
        print(f"WARNING: {filepath} not found — skipping")

first_name = list(chunk_sets.keys())[0]
sample = chunk_sets[first_name][0]
print(f"\nSample chunk fields: {list(sample.keys())}")
print(f"  chunk_id: {sample['chunk_id']}")
print(f"  title: {sample.get('title', 'N/A')}")
print(f"  section: {sample.get('section', 'N/A')}")
print(f"  chunk_text: {sample['chunk_text'][:150]}...")

Loaded section_aware: 1214 chunks
Loaded sliding_fixed: 1206 chunks
Loaded sliding_sentence: 574 chunks

Sample chunk fields: ['chunk_id', 'doc_id', 'title', 'source', 'url', 'section', 'chunk_index', 'word_count', 'chunk_text']
  chunk_id: wikipedia:mediterranean_cuisine_c001
  title: Mediterranean cuisine
  section: Intro
  chunk_text: Mediterranean cuisine is the food and methods of preparation used by the people of the Mediterranean basin. The idea of a Mediterranean cuisine origin...


In [5]:
import json

print("loading rag_benchmark.json ...")
with open("rag_benchmark.json", "r", encoding="utf-8") as f:
    QA_BENCHMARK_FULL = json.load(f)

QA_BENCHMARK = QA_BENCHMARK_FULL

print(f"The number of questions: {len(QA_BENCHMARK)}")

loading rag_benchmark.json ...
The number of questions: 90


In [60]:
# I am taking only the first 30 questions
# QA_BENCHMARK = QA_BENCHMARK[:30]

---
# Component 2: Vectorisation / Embedding

We convert text chunks into numerical vector representations using 3 approaches:

- **Approach A** (`all-MiniLM-L6-v2`, 384-dim): Fast lightweight baseline
- **Approach B** (`all-mpnet-base-v2`, 768-dim): Larger model, captures more nuance
- **Approach C** (`all-mpnet-base-v2` + metadata prefix, 768-dim): Same as B but we prepend
  the chunk's title and section to give the model more context about what each chunk is about

In [6]:
def embed_texts(texts, model_name, batch_size=64):
    """Embed a list of texts using a sentence-transformer model."""
    print(f"  Loading model: {model_name}")
    model = SentenceTransformer(model_name, device=DEVICE)
    print(f"  Embedding {len(texts)} texts...")
    start = time.time()
    embeddings = model.encode(
        texts, batch_size=batch_size, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True,
    )
    elapsed = time.time() - start
    print(f"  Done in {elapsed:.1f}s — shape: {embeddings.shape}")
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return embeddings


def enrich_chunk_text(chunk):
    """Prepend title and section to chunk text for Approach C."""
    parts = []
    if chunk.get("title"):
        parts.append(chunk["title"])
    if chunk.get("section"):
        parts.append(chunk["section"])
    if parts:
        return " | ".join(parts) + ": " + chunk["chunk_text"]
    return chunk["chunk_text"]

print("Original text:")
print(f"  {sample['chunk_text']}...")
print("\nEnriched text:")
print(f"  {enrich_chunk_text(sample)}...")

Original text:
  Mediterranean cuisine is the food and methods of preparation used by the people of the Mediterranean basin. The idea of a Mediterranean cuisine originates with the cookery writer Elizabeth David's A Book of Mediterranean Food (1950), and was amplified by other writers working in English. Many writers define the three core elements of the cuisine as the olive, wheat, and the grape, yielding olive oil, bread and pasta, and wine; other writers deny that the widely varied foods of the Mediterranean basin constitute a cuisine at all. A common definition of the geographical area covered, proposed by David, follows the distribution of the olive tree. The region spans a wide variety of cultures with distinct cuisines, in particular (going anticlockwise around the region) the Maghrebi, Egyptian, Levantine, Ottoman (Turkish), Greek, Italian, French (Provençal), and Spanish, although some authors include additional cuisines. Portuguese cuisine, in particular, is partly Mediterran

In [7]:
EMBEDDING_APPROACHES = {
    "A_minilm": {
        "model": "sentence-transformers/all-MiniLM-L6-v2",
        "use_enriched_text": False,
        "description": "MiniLM baseline (384-dim)",
    },
    "B_mpnet": {
        "model": "sentence-transformers/all-mpnet-base-v2",
        "use_enriched_text": False,
        "description": "MPNet general (768-dim)",
    },
    "C_mpnet_enriched": {
        "model": "sentence-transformers/all-mpnet-base-v2",
        "use_enriched_text": True,
        "description": "MPNet + metadata prefix (768-dim)",
    },
}

In [8]:
# Run all embeddings (3 chunk sets × 3 approaches = 9 configs)

embeddings_store = {}
timing_log = []

for set_name, chunks in chunk_sets.items():
    embeddings_store[set_name] = {}
    print(f"\n{'='*60}")
    print(f"CHUNK SET: {set_name} ({len(chunks)} chunks)")
    print(f"{'='*60}")

    for approach_name, approach_info in EMBEDDING_APPROACHES.items():
        print(f"\n--- {approach_name}: {approach_info['description']} ---")

        # prepare texts — plain or enriched
        if approach_info["use_enriched_text"]:
            texts = [enrich_chunk_text(c) for c in chunks]
            print("  Using enriched text (title/section prepended)")
        else:
            texts = [c["chunk_text"] for c in chunks]

        # embed
        start = time.time()
        embs = embed_texts(texts, approach_info["model"])
        elapsed = time.time() - start

        embeddings_store[set_name][approach_name] = embs
        timing_log.append({
            "chunk_set": set_name, "approach": approach_name,
            "n_chunks": len(chunks), "dims": embs.shape[1],
            "time_s": round(elapsed, 1),
        })

print("\n\nAll embeddings computed!")

# summary
print(f"\n{'Chunk Set':<22} {'Approach':<22} {'Chunks':>7} {'Dims':>6} {'Time':>7}")
for e in timing_log:
    print(f"{e['chunk_set']:<22} {e['approach']:<22} {e['n_chunks']:>7} {e['dims']:>6} {e['time_s']:>6.1f}s")


CHUNK SET: section_aware (1214 chunks)

--- A_minilm: MiniLM baseline (384-dim) ---
  Loading model: sentence-transformers/all-MiniLM-L6-v2
  Embedding 1214 texts...


Batches:   0%|                                                                                  | 0/19 [00:00<?, ?it/s]C:\Users\Hermes\anaconda3\envs\cv-cuda\lib\site-packages\transformers\models\bert\modeling_bert.py:439: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Batches: 100%|█████████████████████████████████████████████████████████████████████████| 19/19 [00:02<00:00,  7.40it/s]


  Done in 2.6s — shape: (1214, 384)

--- B_mpnet: MPNet general (768-dim) ---
  Loading model: sentence-transformers/all-mpnet-base-v2
  Embedding 1214 texts...


Batches: 100%|█████████████████████████████████████████████████████████████████████████| 19/19 [00:07<00:00,  2.57it/s]


  Done in 7.5s — shape: (1214, 768)

--- C_mpnet_enriched: MPNet + metadata prefix (768-dim) ---
  Using enriched text (title/section prepended)
  Loading model: sentence-transformers/all-mpnet-base-v2
  Embedding 1214 texts...


Batches: 100%|█████████████████████████████████████████████████████████████████████████| 19/19 [00:07<00:00,  2.63it/s]


... (9 outputs left out)

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 9/9 [00:05<00:00,  1.52it/s]


  Done in 6.0s — shape: (574, 768)

--- C_mpnet_enriched: MPNet + metadata prefix (768-dim) ---
  Using enriched text (title/section prepended)
  Loading model: sentence-transformers/all-mpnet-base-v2
  Embedding 574 texts...


Batches: 100%|███████████████████████████████████████████████████████████████████████████| 9/9 [00:05<00:00,  1.58it/s]

  Done in 5.7s — shape: (574, 768)


All embeddings computed!

Chunk Set              Approach                Chunks   Dims    Time
section_aware          A_minilm                  1214    384    4.8s
section_aware          B_mpnet                   1214    768    9.9s
section_aware          C_mpnet_enriched          1214    768    9.0s
sliding_fixed          A_minilm                  1206    384    3.1s
sliding_fixed          B_mpnet                   1206    768   12.3s
sliding_fixed          C_mpnet_enriched          1206    768   12.0s
sliding_sentence       A_minilm                   574    384    2.1s
sliding_sentence       B_mpnet                    574    768    7.5s
sliding_sentence       C_mpnet_enriched           574    768    7.3s


# Component 3: Retrieval and Ranking

Given a query, find the 5 most relevant chunks. Three approaches:

- **Approach 1 (Cosine)**: Embed query → cosine similarity vs all chunks → top 5
- **Approach 2 (Cosine + re-ranking)**: Top 50 by cosine → cross-encoder re-scores each pair → top 5
- **Approach 3 (BM25)**: Classic keyword matching — no neural networks, just word overlap

The cross-encoder reads query and chunk *together* as one input, so it understands
whether the chunk actually answers the question, not just whether it's topically similar.

In [9]:
TOP_K = 5           # max chunks
TOP_K_RERANK = 50   # candidates for re-ranking

# embedding models for encoding queries
EMBEDDING_MODEL_MAP = {
    "A_minilm": "sentence-transformers/all-MiniLM-L6-v2",
    "B_mpnet": "sentence-transformers/all-mpnet-base-v2",
    "C_mpnet_enriched": "sentence-transformers/all-mpnet-base-v2",
}

# cache models so we only load each once
_model_cache = {}

def get_embed_model(approach_name):
    model_id = EMBEDDING_MODEL_MAP[approach_name]
    if model_id not in _model_cache:
        print(f"  Loading {model_id}...")
        _model_cache[model_id] = SentenceTransformer(model_id, device=DEVICE)
    return _model_cache[model_id]

# load cross-encoder for re-ranking
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
print(f"Loading cross-encoder: {CROSS_ENCODER_MODEL}")
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL, device=DEVICE)
print("Cross-encoder loaded.")

Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2
Cross-encoder loaded.


In [10]:
def retrieve_cosine(query, chunks, chunk_embeddings, embed_model, top_k=TOP_K):
    """Approach 1: Vanilla cosine similarity — embed query, compare to all chunks."""
    query_emb = embed_model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    sims = cosine_similarity(query_emb, chunk_embeddings)[0]
    top_indices = np.argsort(sims)[::-1][:top_k]
    return [{"chunk": chunks[idx], "score": float(sims[idx]), "rank": i+1}
            for i, idx in enumerate(top_indices)]


def retrieve_reranked(query, chunks, chunk_embeddings, embed_model,
                      top_k_candidates=TOP_K_RERANK, top_k_final=TOP_K):
    """Approach 2: Cosine to get candidates, then cross-encoder re-ranks them."""
    # stage 1: cosine similarity for candidates
    query_emb = embed_model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    sims = cosine_similarity(query_emb, chunk_embeddings)[0]
    candidate_indices = np.argsort(sims)[::-1][:top_k_candidates]

    # stage 2: cross-encoder re-ranking
    pairs = [(query, chunks[idx]["chunk_text"]) for idx in candidate_indices]
    cross_scores = cross_encoder.predict(pairs)
    scored = sorted(zip(candidate_indices, cross_scores), key=lambda x: x[1], reverse=True)

    return [{"chunk": chunks[int(idx)], "score": float(score), "rank": i+1}
            for i, (idx, score) in enumerate(scored[:top_k_final])]


# BM25 indexes — one per chunk set
bm25_indexes = {}
for set_name, chunks in chunk_sets.items():
    tokenized = [c["chunk_text"].lower().split() for c in chunks]
    bm25_indexes[set_name] = BM25Okapi(tokenized)
    print(f"Built BM25 index for {set_name} ({len(chunks)} chunks)")


def retrieve_bm25(query, chunks, bm25_index, top_k=TOP_K):
    """Approach 3: BM25 keyword search — no embeddings needed."""
    query_tokens = query.lower().split()
    scores = bm25_index.get_scores(query_tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [{"chunk": chunks[idx], "score": float(scores[idx]), "rank": i+1}
            for i, idx in enumerate(top_indices)]

Built BM25 index for section_aware (1214 chunks)
Built BM25 index for sliding_fixed (1206 chunks)
Built BM25 index for sliding_sentence (574 chunks)


In [11]:
# retrieval_results[config_key] = [{query, query_id, gold_answer, results}, ...]
retrieval_results = {}

for set_name, chunks in chunk_sets.items():
    print(f"CHUNK SET: {set_name}")

    # embedding-based retrieval (approaches 1 and 2)
    for emb_name in embeddings_store.get(set_name, {}):
        embs = embeddings_store[set_name][emb_name]
        model = get_embed_model(emb_name)

        for qa in QA_BENCHMARK:
            query = qa["query"]

            # approach 1: vanilla cosine
            key = f"{set_name}__{emb_name}__cosine"
            if key not in retrieval_results:
                retrieval_results[key] = []
            results = retrieve_cosine(query, chunks, embs, model)
            retrieval_results[key].append({
                "query_id": qa["query_id"], "query": query,
                "gold_answer": qa["gold_answer"], "results": results,
            })

            # approach 2: cosine + reranking
            key = f"{set_name}__{emb_name}__reranked"
            if key not in retrieval_results:
                retrieval_results[key] = []
            results = retrieve_reranked(query, chunks, embs, model)
            retrieval_results[key].append({
                "query_id": qa["query_id"], "query": query,
                "gold_answer": qa["gold_answer"], "results": results,
            })

        print(f"  Done: {emb_name} (cosine + reranked)")

    # approach 3: BM25
    key = f"{set_name}__BM25__keyword"
    retrieval_results[key] = []
    for qa in QA_BENCHMARK:
        results = retrieve_bm25(qa["query"], chunks, bm25_indexes[set_name])
        retrieval_results[key].append({
            "query_id": qa["query_id"], "query": qa["query"],
            "gold_answer": qa["gold_answer"], "results": results,
        })
    print(f"  Done: BM25")

print(f"\nTotal retrieval configurations: {len(retrieval_results)}")

CHUNK SET: section_aware
  Loading sentence-transformers/all-MiniLM-L6-v2...
  Done: A_minilm (cosine + reranked)
  Loading sentence-transformers/all-mpnet-base-v2...
  Done: B_mpnet (cosine + reranked)
  Done: C_mpnet_enriched (cosine + reranked)
  Done: BM25
CHUNK SET: sliding_fixed
  Done: A_minilm (cosine + reranked)
  Done: B_mpnet (cosine + reranked)
  Done: C_mpnet_enriched (cosine + reranked)
  Done: BM25
CHUNK SET: sliding_sentence
  Done: A_minilm (cosine + reranked)
  Done: B_mpnet (cosine + reranked)
  Done: C_mpnet_enriched (cosine + reranked)
  Done: BM25

Total retrieval configurations: 21



# Component 4: Prompting and Generation

We use **Qwen/Qwen2.5-0.5B-Instruct** with 4 prompt strategies:

 - **XML-structured**: Context wrapped in XML tags for clear parsing, minimal instructions
 - **Guided + system role**: XML structure + system message defining the model as a cuisine expert with strict grounding constraints
 - **Few-shot**: Guided approach + a worked example showing the model what a perfect answer looks like
 - **Chain-of-thought**: Guided approach + numbered reasoning rules + "let's think step by step" to encourage systematic reasoning

 All prompts use XML tags (`<context>`, `<doc>`, `<question>`) to clearly separate the retrieved chunks from the question. This structured format helps the model understand where the context ends and the question begins.

The model is loaded with `repetition_penalty=1.2` to prevent repetitive outputs and a system message is used in approaches B–D to set the model's behaviour.

In [12]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Loading {MODEL_NAME}...")
gen_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
gen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
    trust_remote_code=True,
)
if DEVICE == "cpu":
    gen_model = gen_model.to(DEVICE)

print(f"Model loaded on {DEVICE}.")

Loading Qwen/Qwen2.5-0.5B-Instruct...
Model loaded on cuda.


In [13]:
def build_prompt_xml(query, retrieved_chunks):
    """
    Approach A: XML-structured, minimal instructions.
    Context wrapped in XML tags for clear parsing, but no role or constraints.
    """
    context_parts = []
    for i, c in enumerate(retrieved_chunks):
        text = c["chunk"]["chunk_text"]
        context_parts.append(f'<doc id="{i+1}">\n{text}\n</doc>')
    context = "\n".join(context_parts)

    prompt = f"""Answer the following question based on the provided context.

<context>
{context}
</context>

<question>
{query}
</question>

Answer:"""

    return prompt

def build_prompt_guided(query, retrieved_chunks):
    """
    Approach B: XML structure + system message + role + grounding constraints.
    System message sets the model's behaviour, user message has structured context.
    """
    context_parts = []
    for i, c in enumerate(retrieved_chunks):
        text = c["chunk"]["chunk_text"]
        context_parts.append(f'<doc id="{i+1}">\n{text}\n</doc>')
    context = "\n".join(context_parts)

    prompt = f"""Answer the question using ONLY the information in the context below.
If the context does not contain enough information, say "I don't have enough information to answer this."
Keep your answer concise and factual.

<context>
{context}
</context>

<question>
{query}
</question>

Answer:"""

    return prompt

def build_prompt_fewshot(query, retrieved_chunks):
    """
    Approach C: Guided approach + a worked example before the real question.
    The example teaches the model the expected answer format and style.
    """
    context_parts = []
    for i, c in enumerate(retrieved_chunks):
        text = c["chunk"]["chunk_text"]
        context_parts.append(f'<doc id="{i+1}">\n{text}\n</doc>')
    context = "\n".join(context_parts)

    prompt = f"""Answer the question using ONLY the information in the context below.
If the context does not contain enough information, say "I don't have enough information."

Here is an example:

<context>
<doc id="1">
Olive oil has been a cornerstone of Mediterranean cooking for over 5,000 years. The Mediterranean region accounts for the world's highest consumption of olive oil. In 2014, Greece used 17 kg per head, Italy 12 kg, and Spain 13 kg.
</doc>
</context>

<question>
Which Mediterranean country consumes the most olive oil per person?
</question>

Answer: Greece consumes the most olive oil per person in the Mediterranean, using 17 kg per head as of 2014, followed by Spain at 13 kg and Italy at 12 kg.

Now answer the following:

<context>
{context}
</context>

<question>
{query}
</question>

Answer:"""

    return prompt


def build_prompt_cot(query, retrieved_chunks):
    """
    Approach D: Chain-of-thought — strict rules + "let's think step by step".
    Encourages the model to reason through the context before answering.
    """
    context_parts = []
    for i, c in enumerate(retrieved_chunks):
        text = c["chunk"]["chunk_text"]
        context_parts.append(f'<doc id="{i+1}">\n{text}\n</doc>')
    context = "\n".join(context_parts)

    prompt = f"""You must follow these rules strictly:
1. Base your answer ONLY on the provided context documents
2. Do NOT use your pre-trained knowledge or guess information
3. If the documents do not contain the answer, say "I cannot answer this based on the provided documents"
4. Keep your answer concise and factual

<context>
{context}
</context>

<question>
{query}
</question>

Let's think step by step to find the answer from the context:"""

    return prompt


PROMPT_STRATEGIES = {
    "xml": build_prompt_xml,
    "guided": build_prompt_guided,
    "fewshot": build_prompt_fewshot,
    "cot": build_prompt_cot,
}

In [14]:
SYSTEM_MESSAGES = {
    "xml": None,  # no system message for the simple XML approach
    "guided": "You are an expert Mediterranean cuisine assistant. Answer strictly based on the provided context.",
    "fewshot": "You are an expert Mediterranean cuisine assistant. Answer strictly based on the provided context.",
    "cot": (
        "You are an expert culinary AI assistant specialized in Mediterranean cuisine and food history. "
        "You are strictly bound by the context documents provided. Do not use outside knowledge."
    ),
}


def generate_answer(query, retrieved_chunks, prompt_fn, prompt_name="xml", max_new_tokens=256):
    """
    Generate an answer using Qwen2.5-0.5B-Instruct.
    Uses system message for guided/fewshot/cot approaches.
    Uses repetition_penalty to prevent repeated phrases.
    """
    prompt = prompt_fn(query, retrieved_chunks)

    # build chat messages — with or without system message
    messages = []
    sys_msg = SYSTEM_MESSAGES.get(prompt_name)
    if sys_msg:
        messages.append({"role": "system", "content": sys_msg})
    messages.append({"role": "user", "content": prompt})

    text = gen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = gen_tokenizer([text], return_tensors="pt").to(gen_model.device)

    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.2,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    answer = gen_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return answer

In [70]:
#SKIP for demo

# Run generation for ALL configurations
# (21 retrieval configs × 4 prompt strategies)

generation_results = {}

# total = len(retrieval_results) * len(PROMPT_STRATEGIES)
done = 0

for ret_key, ret_entries in retrieval_results.items():
    for prompt_name, prompt_fn in PROMPT_STRATEGIES.items():
        gen_key = f"{ret_key}__{prompt_name}"
        generation_results[gen_key] = []
        i = 0
        total = len(ret_entries)

        for entry in ret_entries:
            i += 1
            answer = generate_answer(
                entry["query"], entry["results"], prompt_fn,
                prompt_name=prompt_name,  # passes the name so we pick the right system message
            )

            print(f"[{i}/{total}] entries done")

            generation_results[gen_key].append({
                "query_id": entry["query_id"],
                "query": entry["query"],
                "gold_answer": entry["gold_answer"],
                "response": answer,
                "retrieved_context": [
                    {"doc_id": r["chunk"]["chunk_id"], "text": r["chunk"]["chunk_text"]}
                    for r in entry["results"]
                ],
            })

        done += 1
        print(f"############# [{done}] CONFIG(s) DONE #############")

print(f"\nTotal generation configs: {len(generation_results)}")
print(f"Total answers generated: {sum(len(v) for v in generation_results.values())}")

[1/90] entries done
[2/90] entries done
[3/90] entries done
[4/90] entries done
[5/90] entries done
[6/90] entries done
[7/90] entries done
[8/90] entries done
[9/90] entries done
[10/90] entries done
[11/90] entries done
[12/90] entries done
[13/90] entries done
[14/90] entries done
[15/90] entries done
[16/90] entries done
[17/90] entries done
[18/90] entries done
[19/90] entries done
[20/90] entries done
[21/90] entries done
[22/90] entries done
[23/90] entries done
[24/90] entries done
[25/90] entries done
[26/90] entries done
[27/90] entries done
[28/90] entries done
[29/90] entries done
[30/90] entries done
[31/90] entries done
[32/90] entries done
[33/90] entries done
[34/90] entries done
[35/90] entries done
[36/90] entries done
[37/90] entries done
[38/90] entries done
[39/90] entries done
[40/90] entries done
[41/90] entries done
[42/90] entries done
[43/90] entries done
[44/90] entries done
[45/90] entries done
[46/90] entries done
[47/90] entries done
[48/90] entries done
[

In [71]:
with open("generation_results.json", "w", encoding="utf-8") as f:
    json.dump(generation_results, f, indent=2, ensure_ascii=False)

print(f"Saved generation_results.json ({len(generation_results)} configs, "
      f"{sum(len(v) for v in generation_results.values())} total answers)")


with open("generation_results.json", "r", encoding="utf-8") as f:
    generation_results = json.load(f)

Saved generation_results.json (84 configs, 7560 total answers)


# Evaluation

In [15]:
from rouge_score import rouge_scorer
from collections import Counter
import numpy as np

In [16]:
# ---------- RETRIEVAL METRICS ----------

def chunk_contains_answer(chunk_text, gold_answer, threshold=0.3):
    """
    Check if a chunk contains enough keywords from the gold answer.
    We remove common stopwords and check what fraction of the
    gold answer's meaningful words appear in the chunk.
    """
    stopwords = {"the", "a", "an", "is", "are", "was", "were", "of", "in",
                 "to", "and", "for", "on", "it", "that", "this", "with",
                 "from", "as", "by", "at", "be", "its", "or", "not"}

    chunk_tokens = set(chunk_text.lower().split())
    answer_tokens = set(gold_answer.lower().split()) - stopwords

    if not answer_tokens:
        return False

    overlap = chunk_tokens & answer_tokens
    return len(overlap) / len(answer_tokens) >= threshold


def recall_at_k(retrieved_chunks, gold_answer, k=5):
    """
    Recall@K: 1 if any of the top-k chunks contains the answer, else 0.
    This tells us whether the retrieval step found the right information.
    """
    for r in retrieved_chunks[:k]:
        chunk_text = r["chunk"]["chunk_text"] if "chunk" in r else r.get("text", "")
        if chunk_contains_answer(chunk_text, gold_answer):
            return 1.0
    return 0.0

def mrr_score(retrieved_chunks, gold_answer):
    """
    Mean Reciprocal Rank: 1/rank of the first relevant chunk.
    Rank 1 = 1.0, Rank 2 = 0.5, Rank 3 = 0.33, etc.
    If no relevant chunk found, returns 0.
    """
    for i, r in enumerate(retrieved_chunks):
        chunk_text = r["chunk"]["chunk_text"] if "chunk" in r else r.get("text", "")
        if chunk_contains_answer(chunk_text, gold_answer):
            return 1.0 / (i + 1)
    return 0.0


In [17]:
# ---------- GENERATION METRICS ----------

def compute_token_f1(prediction, gold):
    """
    Token-level F1: treat both answers as bags of words.
    Precision = what fraction of predicted words are in gold
    Recall = what fraction of gold words are in prediction
    F1 = harmonic mean
    """
    pred_tokens = prediction.lower().split()
    gold_tokens = gold.lower().split()

    if not pred_tokens or not gold_tokens:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_common = sum(common.values())

    if num_common == 0:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    precision = num_common / len(pred_tokens)
    recall = num_common / len(gold_tokens)
    f1 = 2 * precision * recall / (precision + recall)
    return {"precision": precision, "recall": recall, "f1": f1}


def compute_rouge_l(prediction, gold):
    """
    ROUGE-L: longest common subsequence between prediction and gold.
    Higher = more overlap in word ordering.
    """
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scores = scorer.score(gold, prediction)
    return scores["rougeL"].fmeasure


def compute_bert_scores(predictions, golds):
    """
    BERTScore (proxy): cosine similarity between sentence embeddings.
    Catches semantic similarity even when wording differs.
    e.g. "prevents crystallization" vs "keeps it from crystallizing" = high score
    """
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
    pred_embs = model.encode(predictions, normalize_embeddings=True, convert_to_numpy=True)
    gold_embs = model.encode(golds, normalize_embeddings=True, convert_to_numpy=True)
    scores = [float(np.dot(p, g)) for p, g in zip(pred_embs, gold_embs)]
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return scores

In [75]:
eval_results = []

for gen_key, entries in generation_results.items():

    # skip configs with no entries
    if not entries:
        continue

    # collect per-question scores
    recalls = []
    mrrs = []
    rouges = []
    f1s = []
    predictions = []
    golds = []

    for entry in entries:
        gold = entry["gold_answer"]
        pred = entry["response"] if entry["response"].strip() else ""
        retrieved = entry.get("retrieved_context", [])

        # retrieval metrics
        recalls.append(recall_at_k(retrieved, gold))
        mrrs.append(mrr_score(retrieved, gold))

        # generation metrics
        if pred:
            rouges.append(compute_rouge_l(pred, gold))
            f1s.append(compute_token_f1(pred, gold)["f1"])
        else:
            rouges.append(0.0)
            f1s.append(0.0)

        predictions.append(pred if pred else "no answer")
        golds.append(gold)

    # BERTScore (batch)
    bert_scores = compute_bert_scores(predictions, golds)

    eval_results.append({
        "config": gen_key,
        "n_questions": len(entries),
        "recall_at_5": float(np.mean(recalls)),
        "mrr": float(np.mean(mrrs)),
        "rouge_l": float(np.mean(rouges)),
        "token_f1": float(np.mean(f1s)),
        "bert_score": float(np.mean(bert_scores)),
        "answered": sum(1 for e in entries if e["response"].strip()),
    })

print(f"Evaluated {len(eval_results)} configurations")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6056.54it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11450.13it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12482.33it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embe

Evaluated 84 configurations


In [76]:
# sort by rouge_l descending
eval_results.sort(key=lambda x: x["rouge_l"], reverse=True)

print(f"{'Configuration':<60} {'Recall@5':>8} {'MRR':>6} {'ROUGE-L':>8} {'F1':>6} {'BERT':>6} {'Ans':>4}")

for r in eval_results:
    print(f"{r['config']:<60} {r['recall_at_5']:>8.3f} {r['mrr']:>6.3f} "
          f"{r['rouge_l']:>8.3f} {r['token_f1']:>6.3f} {r['bert_score']:>6.3f} "
          f"{r['answered']:>3}/{r['n_questions']}")

Configuration                                                Recall@5    MRR  ROUGE-L     F1   BERT  Ans
section_aware__C_mpnet_enriched__cosine__guided                 0.878  0.742    0.342  0.315  0.815  90/90
section_aware__C_mpnet_enriched__reranked__guided               0.900  0.779    0.325  0.306  0.810  90/90
section_aware__B_mpnet__cosine__guided                          0.867  0.770    0.323  0.309  0.813  90/90
sliding_fixed__A_minilm__reranked__guided                       0.978  0.889    0.323  0.309  0.817  90/90
sliding_fixed__C_mpnet_enriched__reranked__guided               1.000  0.916    0.321  0.309  0.827  90/90
sliding_fixed__C_mpnet_enriched__cosine__guided                 0.933  0.780    0.317  0.313  0.807  90/90
section_aware__A_minilm__cosine__guided                         0.844  0.732    0.315  0.309  0.819  90/90
sliding_fixed__B_mpnet__reranked__fewshot                       1.000  0.916    0.314  0.316  0.820  90/90
sliding_fixed__B_mpnet__cosine__guided 

In [77]:

print("TOP 5 CONFIGURATIONS (by ROUGE-L)")

for i, r in enumerate(eval_results[:5], 1):
    print(f"  {i:2d}. ROUGE-L={r['rouge_l']:.3f} | Recall@5={r['recall_at_5']:.2f} | "
          f"MRR={r['mrr']:.3f} | F1={r['token_f1']:.3f} | BERT={r['bert_score']:.3f}")
    print(f"      {r['config']}")


print("WORST 5 CONFIGURATIONS")

for r in eval_results[-5:]:
    print(f"  ROUGE-L={r['rouge_l']:.3f} | Recall@5={r['recall_at_5']:.2f}")
    print(f"      {r['config']}")


TOP 5 CONFIGURATIONS (by ROUGE-L)
   1. ROUGE-L=0.342 | Recall@5=0.88 | MRR=0.742 | F1=0.315 | BERT=0.815
      section_aware__C_mpnet_enriched__cosine__guided
   2. ROUGE-L=0.325 | Recall@5=0.90 | MRR=0.779 | F1=0.306 | BERT=0.810
      section_aware__C_mpnet_enriched__reranked__guided
   3. ROUGE-L=0.323 | Recall@5=0.87 | MRR=0.770 | F1=0.309 | BERT=0.813
      section_aware__B_mpnet__cosine__guided
   4. ROUGE-L=0.323 | Recall@5=0.98 | MRR=0.889 | F1=0.309 | BERT=0.817
      sliding_fixed__A_minilm__reranked__guided
   5. ROUGE-L=0.321 | Recall@5=1.00 | MRR=0.916 | F1=0.309 | BERT=0.827
      sliding_fixed__C_mpnet_enriched__reranked__guided
WORST 5 CONFIGURATIONS
  ROUGE-L=0.154 | Recall@5=0.83
      sliding_sentence__B_mpnet__reranked__cot
  ROUGE-L=0.151 | Recall@5=0.82
      sliding_sentence__C_mpnet_enriched__cosine__cot
  ROUGE-L=0.149 | Recall@5=0.57
      sliding_fixed__BM25__keyword__cot
  ROUGE-L=0.144 | Recall@5=0.56
      section_aware__BM25__keyword__cot
  ROUGE-L=0.143

In [78]:
# Per-question best results

question_summary = []

for qa in QA_BENCHMARK:
    qid = qa["query_id"]
    gold = qa["gold_answer"]

    best_rouge = -1
    best_answer = ""
    best_config = ""
    total_configs_with_answer = 0

    for gen_key, entries in generation_results.items():
        for entry in entries:
            if entry["query_id"] == qid and entry["response"].strip():
                total_configs_with_answer += 1
                rouge = compute_rouge_l(entry["response"], gold)
                if rouge > best_rouge:
                    best_rouge = rouge
                    best_answer = entry["response"]
                    best_config = gen_key

    status = "GOOD" if best_rouge > 0.5 else "WEAK" if best_rouge > 0.2 else "FAIL"

    question_summary.append({
        "qid": qid,
        "query": qa["query"],
        "gold": gold,
        "best_answer": best_answer,
        "best_rouge": round(best_rouge, 3),
        "best_config": best_config,
        "configs_answered": total_configs_with_answer,
        "status": status,
    })

# save to JSON
with open("question_summary.json", "w", encoding="utf-8") as f:
    json.dump(question_summary, f, indent=2, ensure_ascii=False)


print(f"\n{'QID':<5} {'Status':<6} {'ROUGE':>6} {'Cfgs':>5}  {'Query':<40} {'Best Answer':<50}")
print("-" * 115)
for q in question_summary[:20]:
    qr = q["query"][:38] + ".." if len(q["query"]) > 40 else q["query"]
    ans = q["best_answer"][:48] + ".." if len(q["best_answer"]) > 50 else q["best_answer"]
    print(f"{q['qid']:<5} {q['status']:<6} {q['best_rouge']:>6.3f} {q['configs_answered']:>5}  {qr:<40} {ans:<50}")

# distribution
good = sum(1 for q in question_summary if q["status"] == "GOOD")
weak = sum(1 for q in question_summary if q["status"] == "WEAK")
fail = sum(1 for q in question_summary if q["status"] == "FAIL")
print(f"\n")
print(f"Total: {len(question_summary)} questions")
print(f"GOOD (>0.5): {good}  |  WEAK (0.2-0.5): {weak}  |  FAIL (<0.2): {fail}")

# Worst questions — where should we focus improvement?
print("\n\nHARDEST QUESTIONS (lowest best ROUGE-L)")
for q in sorted(question_summary, key=lambda x: x["best_rouge"])[:10]:
    print(f"\n  Q[{q['qid']}] ROUGE={q['best_rouge']:.3f}: {q['query'][:70]}...")
    print(f"    Gold:   {q['gold'][:100]}...")
    print(f"    Got:    {q['best_answer'][:100]}...")
    print(f"    Config: {q['best_config']}")


QID   Status  ROUGE  Cfgs  Query                                    Best Answer                                       
-------------------------------------------------------------------------------------------------------------------
01    GOOD    0.615    84  What are the ingredients in Algerian B.. The main ingredients for Algerian Baghrir are:

..
02    GOOD    0.800    84  What are the main ingredients in Baba .. The main ingredients in Baba Ganoush are:

- Med..
03    GOOD    0.588    84  What are the ingredients needed to mak.. The main ingredient required to make Tzatziki is..
04    GOOD    0.639    84  What are the ingredients for Shakshuka.. The ingredients required for making Shakshuka I ..
05    WEAK    0.482    84  What are the ingredients for Gazpacho?   Based on the information provided:

Gazpacho Rec..
06    GOOD    0.609    84  What are the ingredients in the Sangri.. In the Sangria recipe:

- Red wine 
- Peaches  
..
07    GOOD    0.545    84  What are the ingredient

In [79]:
best_config = eval_results[0]["config"]
best_entries = generation_results[best_config]

print(f"BEST CONFIG: {best_config}")

BEST CONFIG: section_aware__C_mpnet_enriched__cosine__guided


In [80]:
print("ANALYSIS: Average scores by component choice")

# helper to extract component from config key
def get_component(config_key, component):
    parts = config_key.split("__")
    if component == "chunk_set":
        return parts[0]
    elif component == "embedding":
        return parts[1] if len(parts) > 1 else "BM25"
    elif component == "retrieval":
        return parts[2] if len(parts) > 2 else "keyword"
    elif component == "prompt":
        return parts[3] if len(parts) > 3 else "unknown"
    return "unknown"

for component_name in ["chunk_set", "embedding", "retrieval", "prompt"]:
    print(f"\n--- By {component_name} ---")

    # group results by this component
    groups = {}
    for r in eval_results:
        val = get_component(r["config"], component_name)
        if val not in groups:
            groups[val] = []
        groups[val].append(r)

    print(f"  {'Value':<25} {'ROUGE-L':>8} {'Recall@5':>9} {'MRR':>6} {'Token F1':>9}")
    for val, results in sorted(groups.items()):
        avg_rouge = np.mean([r["rouge_l"] for r in results])
        avg_recall = np.mean([r["recall_at_5"] for r in results])
        avg_mrr = np.mean([r["mrr"] for r in results])
        avg_f1 = np.mean([r["token_f1"] for r in results])
        print(f"  {val:<25} {avg_rouge:>8.3f} {avg_recall:>9.3f} {avg_mrr:>6.3f} {avg_f1:>9.3f}")


ANALYSIS: Average scores by component choice

--- By chunk_set ---
  Value                      ROUGE-L  Recall@5    MRR  Token F1
  section_aware                0.252     0.833  0.718     0.252
  sliding_fixed                0.255     0.908  0.797     0.259
  sliding_sentence             0.211     0.789  0.696     0.222

--- By embedding ---
  Value                      ROUGE-L  Recall@5    MRR  Token F1
  A_minilm                     0.244     0.885  0.781     0.248
  BM25                         0.201     0.567  0.453     0.217
  B_mpnet                      0.246     0.889  0.791     0.249
  C_mpnet_enriched             0.247     0.894  0.781     0.249

--- By retrieval ---
  Value                      ROUGE-L  Recall@5    MRR  Token F1
  cosine                       0.244     0.869  0.747     0.247
  keyword                      0.201     0.567  0.453     0.217
  reranked                     0.248     0.910  0.822     0.250

--- By prompt ---
  Value                      ROUGE-L  

# Demo Inference Pipeline

In [18]:

# section_aware chunks + C_mpnet_enriched + cosine + guided prompt

BEST_CHUNKS = chunk_sets["section_aware"]
BEST_EMBS = embeddings_store["section_aware"]["C_mpnet_enriched"]
BEST_EMBED_MODEL = get_embed_model("C_mpnet_enriched")

def run_live_demo(input_payload):
    """
    Full pipeline for the demo.
    Input:  {"queries": [{"query_id": "0", "query": "..."}]}
    Output: {"results": [{"query_id": "0", "query": "...", "response": "...", "retrieved_context": [...]}]}
    """
    results = []
    for q in input_payload["queries"]:
        print(f"  Processing: {q['query'][:50]}...")

        # retrieve with cosine
        retrieved = retrieve_cosine(q["query"], BEST_CHUNKS, BEST_EMBS, BEST_EMBED_MODEL)

        # generate with guided prompt
        answer = generate_answer(q["query"], retrieved, build_prompt_guided)

        results.append({
            "query_id": q["query_id"],
            "query": q["query"],
            "response": answer,
            "retrieved_context": [
                {"doc_id": r["chunk"]["chunk_id"], "text": r["chunk"]["chunk_text"]}
                for r in retrieved
            ],
        })
    return {"results": results}

In [19]:
# Upload the test input file
print("Upload your input_payload.json:")
INPUT_FILE = "input_payload.json"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    test_input = json.load(f)

print(f"Loaded {len(test_input['queries'])} queries\n")

# run pipeline
output = run_live_demo(test_input)

# save output
OUTPUT_FILE = "output_payload.json"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"\nSaved {OUTPUT_FILE} ({len(output['results'])} results)")

# preview first result
print("\nFirst result:")
print(json.dumps(output["results"][0], indent=2, ensure_ascii=False))



Upload your input_payload.json:
Loaded 2 queries

  Processing: What's the longest river in the world?...


C:\Users\Hermes\anaconda3\envs\cv-cuda\lib\site-packages\transformers\generation\configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
C:\Users\Hermes\anaconda3\envs\cv-cuda\lib\site-packages\transformers\generation\configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
C:\Users\Hermes\anaconda3\envs\cv-cuda\lib\site-packages\transformers\generation\configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  Processing: What does the democratic republic of Congo flag re...


C:\Users\Hermes\anaconda3\envs\cv-cuda\lib\site-packages\transformers\generation\configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
C:\Users\Hermes\anaconda3\envs\cv-cuda\lib\site-packages\transformers\generation\configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
C:\Users\Hermes\anaconda3\envs\cv-cuda\lib\site-packages\transformers\generation\configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(



Saved output_payload.json (2 results)

First result:
{
  "query_id": "0",
  "query": "What's the longest river in the world?",
  "response": "I don't have enough information to answer this question.",
  "retrieved_context": [
    {
      "doc_id": "around80_spain_04_c003",
      "text": "I next added green peas, rings of squid, prawns, clams, and mussels in order of required cooking time, then left the liquid to absorb uncovered over a medium-low heat. I served my paella with slices of lemon and sprigs of fresh parsley, and I cooked it over the barbecue in a paella pan I picked up at a second hand store, which is required to be shallow and wide, to assist in the formation of the socarrat, perhaps. Although my paella pan is more than double the size of any of my other saucepans, it pales in comparison to some of the largest, which have made it into the Guinness World Records after feeding over 100,000 people."
    },
    {
      "doc_id": "wikipedia:paella_c012",
      "text": "Paella 

# Demo evaluation Pipeline

In [ ]:
BEST_CONFIG_KEY = "section_aware__C_mpnet_enriched__cosine__guided"

generation_results = {BEST_CONFIG_KEY: []}

for qa in QA_BENCHMARK:
    retrieved = retrieve_cosine(
        qa["query"],
        BEST_CHUNKS,
        BEST_EMBS,
        BEST_EMBED_MODEL,
        top_k=5
    )

    answer = generate_answer(
        qa["query"],
        retrieved,
        build_prompt_guided,
        prompt_name="guided"
    )

    generation_results[BEST_CONFIG_KEY].append({
        "query_id": qa["query_id"],
        "query": qa["query"],
        "gold_answer": qa["gold_answer"],
        "response": answer,
        "retrieved_context": [
            {
                "doc_id": r["chunk"]["chunk_id"],
                "text": r["chunk"]["chunk_text"]
            }
            for r in retrieved
        ],
    })

print("Configs in generation_results:", list(generation_results.keys()))
print("Answers generated:", len(generation_results[BEST_CONFIG_KEY]))

In [22]:
eval_results = []

for gen_key, entries in generation_results.items():

    # skip configs with no entries
    if not entries:
        continue

    # collect per-question scores
    recalls = []
    mrrs = []
    rouges = []
    f1s = []
    predictions = []
    golds = []

    for entry in entries:
        gold = entry["gold_answer"]
        pred = entry["response"] if entry["response"].strip() else ""
        retrieved = entry.get("retrieved_context", [])

        # retrieval metrics
        recalls.append(recall_at_k(retrieved, gold))
        mrrs.append(mrr_score(retrieved, gold))

        # generation metrics
        if pred:
            rouges.append(compute_rouge_l(pred, gold))
            f1s.append(compute_token_f1(pred, gold)["f1"])
        else:
            rouges.append(0.0)
            f1s.append(0.0)

        predictions.append(pred if pred else "no answer")
        golds.append(gold)

    # BERTScore (batch)
    bert_scores = compute_bert_scores(predictions, golds)

    eval_results.append({
        "config": gen_key,
        "n_questions": len(entries),
        "recall_at_5": float(np.mean(recalls)),
        "mrr": float(np.mean(mrrs)),
        "rouge_l": float(np.mean(rouges)),
        "token_f1": float(np.mean(f1s)),
        "bert_score": float(np.mean(bert_scores)),
        "answered": sum(1 for e in entries if e["response"].strip()),
    })


Evaluated 1 configurations


In [25]:
# sort by rouge_l descending
eval_results.sort(key=lambda x: x["rouge_l"], reverse=True)

print(f"{'Configuration':<60} {'Recall@5':>8} {'MRR':>6} {'ROUGE-L':>8} {'F1':>6} {'BERT':>6} {'Ans':>4}")

for r in eval_results:
    print(f"{r['config']:<60} {r['recall_at_5']:>8.3f} {r['mrr']:>6.3f} "
          f"{r['rouge_l']:>8.3f} {r['token_f1']:>6.3f} {r['bert_score']:>6.3f} "
          f"{r['answered']:>3}/{r['n_questions']}")

Configuration                                                Recall@5    MRR  ROUGE-L     F1   BERT  Ans
section_aware__C_mpnet_enriched__cosine__guided                 0.878  0.742    0.339  0.315  0.809  90/90


In [27]:
# Per-question best results

question_summary = []

for qa in QA_BENCHMARK:
    qid = qa["query_id"]
    gold = qa["gold_answer"]

    best_rouge = -1
    best_answer = ""
    best_config = ""
    total_configs_with_answer = 0

    for gen_key, entries in generation_results.items():
        for entry in entries:
            if entry["query_id"] == qid and entry["response"].strip():
                total_configs_with_answer += 1
                rouge = compute_rouge_l(entry["response"], gold)
                if rouge > best_rouge:
                    best_rouge = rouge
                    best_answer = entry["response"]
                    best_config = gen_key

    status = "GOOD" if best_rouge > 0.5 else "WEAK" if best_rouge > 0.2 else "FAIL"

    question_summary.append({
        "qid": qid,
        "query": qa["query"],
        "gold": gold,
        "best_answer": best_answer,
        "best_rouge": round(best_rouge, 3),
        "best_config": best_config,
        "configs_answered": total_configs_with_answer,
        "status": status,
    })

# save to JSON
with open("question_summary.json", "w", encoding="utf-8") as f:
    json.dump(question_summary, f, indent=2, ensure_ascii=False)


print(f"\n{'QID':<5} {'Status':<6} {'ROUGE':>6} {'Cfgs':>5}  {'Query':<40} {'Best Answer':<50}")
print("-" * 115)
for q in question_summary[:20]:
    qr = q["query"][:38] + ".." if len(q["query"]) > 40 else q["query"]
    ans = q["best_answer"][:48] + ".." if len(q["best_answer"]) > 50 else q["best_answer"]
    print(f"{q['qid']:<5} {q['status']:<6} {q['best_rouge']:>6.3f} {q['configs_answered']:>5}  {qr:<40} {ans:<50}")

# distribution
good = sum(1 for q in question_summary if q["status"] == "GOOD")
weak = sum(1 for q in question_summary if q["status"] == "WEAK")
fail = sum(1 for q in question_summary if q["status"] == "FAIL")
print(f"\n")
print(f"Total: {len(question_summary)} questions")
print(f"GOOD (>0.5): {good}  |  WEAK (0.2-0.5): {weak}  |  FAIL (<0.2): {fail}")

# Worst questions — where should we focus improvement?
print("\n\nHARDEST QUESTIONS (lowest best ROUGE-L)")
for q in sorted(question_summary, key=lambda x: x["best_rouge"])[:10]:
    print(f"\n  Q[{q['qid']}] ROUGE={q['best_rouge']:.3f}: {q['query'][:70]}...")
    print(f"    Gold:   {q['gold'][:100]}...")
    print(f"    Got:    {q['best_answer'][:100]}...")
    print(f"    Config: {q['best_config']}")


QID   Status  ROUGE  Cfgs  Query                                    Best Answer                                       
-------------------------------------------------------------------------------------------------------------------
01    WEAK    0.486     1  What are the ingredients in Algerian B.. The main ingredients for Algerian Baghrir are:

..
02    GOOD    0.644     1  What are the main ingredients in Baba .. The main ingredients in Baba Ganoush are:

1. Me..
03    WEAK    0.338     1  What are the ingredients needed to mak.. The main ingredient required to make Tzatziki is..
04    GOOD    0.598     1  What are the ingredients for Shakshuka.. The ingredients for Shakshuka include:

- 6 tbsp..
05    WEAK    0.427     1  What are the ingredients for Gazpacho?   To make Gazpacho:

- 2 oz (50 g) 2–3 day-old whi..
06    WEAK    0.429     1  What are the ingredients in the Sangri.. The ingredients in the Sangria recipe include:

..
07    WEAK    0.448     1  What are the ingredient